# Session 8 — Streaming Responses

Streaming is one of the key differences between a static notebook demo and a responsive application. This notebook shows the event pattern we will later reuse in Gradio and Streamlit.

## Learning Goals

- understand why streaming improves app experience
- use the Responses API with `stream=True`
- process text deltas from the event stream
- connect the same pattern to later UI examples
- compare streaming patterns across OpenAI and Ollama


In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_ORG_ID = os.getenv('OPENAI_ORG_ID')
OPENAI_PROJECT_ID = os.getenv('OPENAI_PROJECT_ID')
print('OpenAI key present:', bool(OPENAI_API_KEY))
print('OpenAI org ID present:', bool(OPENAI_ORG_ID))
print('OpenAI project ID present:', bool(OPENAI_PROJECT_ID))

GENERATION_MODEL = os.getenv("SESSION8_MODEL", "gpt-4.1-mini")


OpenAI key present: True
OpenAI org ID present: True
OpenAI project ID present: True


## Non-Streaming Baseline

Start with a normal Response API call so we can compare it to the streaming version.

In [2]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

response = client.responses.create(
    model=GENERATION_MODEL,
    instructions='You are a concise teaching assistant.',
    input='Explain in two short sentences why streaming is useful in chat apps.'
)

display(Markdown(response.output_text))

Streaming in chat apps enables real-time message delivery, creating seamless and instant communication. It also reduces server load by maintaining persistent connections instead of repeated requests.

## Streaming with the Responses API

Now we ask the same style of question, but receive output incrementally as events arrive.

In [3]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

response_stream = client.responses.create(
    model=GENERATION_MODEL,
    instructions='You are a concise teaching assistant.',
    input='Explain in two short sentences why streaming is useful in chat apps.',
    stream=True,
)

streamed_text = ''
handle = display(Markdown('_Streaming response will appear here..._'), display_id=True)

for event in response_stream:
    if event.type == 'response.output_text.delta':
        streamed_text += event.delta
        handle.update(Markdown(streamed_text))

Streaming allows real-time message delivery, enabling instant communication between users. It also reduces latency and improves the chat app's responsiveness.

## Streaming with Ollama

Ollama can expose an OpenAI-compatible Responses API. That means the same event pattern can work against a local model server, as long as you already have a compatible model running locally.

Requires Ollama 0.13.3 or later for its stateless Responses API. Pull `llama3.2:latest` first; set `OLLAMA_CHAT_MODEL` to use another installed model.


In [4]:
# Optional: Ollama streaming with the OpenAI SDK and the Responses API
# This requires Ollama running locally and a model already pulled,
# for example: `ollama pull llama3.2:latest`.

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
OLLAMA_CHAT_MODEL = os.getenv('OLLAMA_CHAT_MODEL', 'llama3.2:latest')

ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
ollama_stream = ollama_client.responses.create(
    model=OLLAMA_CHAT_MODEL,
    input='Explain in two short sentences why streaming is useful in chat apps.',
    stream=True,
)

ollama_text = ''
handle = display(Markdown('_Ollama streaming response will appear here..._'), display_id=True)

for event in ollama_stream:
    if event.type == 'response.output_text.delta':
        ollama_text += event.delta
        handle.update(Markdown(ollama_text))


Streaming in chat apps allows for real-time audio or video transmission, enabling seamless voice or video conversations without the need for downloads or buffering. This feature provides a more engaging and interactive experience for users, making it easier to connect and communicate with others.

## Streaming with Chat Completions

Compare OpenAI Responses events with Chat Completions deltas. This replaces the retired GitHub Models example. Empty housekeeping chunks may have no choices.


In [5]:
chat_client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)
chat_stream = chat_client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[{"role": "user", "content": "Explain in two short sentences why streaming is useful in chat apps."}],
    stream=True,
)
chat_text = ""
handle = display(Markdown("_Streaming..._"), display_id=True)
for chunk in chat_stream:
    if chunk.choices and chunk.choices[0].delta.content:
        chat_text += chunk.choices[0].delta.content
        handle.update(Markdown(chat_text))


Streaming in chat apps enables real-time message delivery, ensuring users receive updates instantly without delay. It also reduces server load by maintaining continuous connections instead of frequent polling.

Streaming is the common idea; the event format depends on the API. Responses emits typed events, while Chat Completions emits choices and content deltas.


## Turn the Event Loop into a Generator

UI frameworks often want a generator or iterable. That makes streaming a natural fit.

In [6]:
def stream_response(prompt: str, model: str = GENERATION_MODEL):
    client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)
    stream = client.responses.create(
        model=model,
        instructions='You are a concise teaching assistant.',
        input=prompt,
        stream=True,
    )

    for event in stream:
        if event.type == 'response.output_text.delta':
            yield event.delta

In [7]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

collected = []
handle = display(Markdown('_Streaming response will appear here..._'), display_id=True)

for piece in stream_response('Give one short paragraph on why streamed output feels faster to users.'):
    collected.append(piece)
    handle.update(Markdown(''.join(collected)))

full_text = ''.join(collected)

display(Markdown('### Final combined text'))
display(Markdown(full_text))

Streamed output feels faster to users because it provides immediate, incremental feedback rather than making them wait for the entire response. This continuous flow of information creates the impression of quick progress and keeps users engaged, reducing perceived latency and enhancing the overall user experience.

### Final combined text

Streamed output feels faster to users because it provides immediate, incremental feedback rather than making them wait for the entire response. This continuous flow of information creates the impression of quick progress and keeps users engaged, reducing perceived latency and enhancing the overall user experience.

## Why This Matters for the Rest of Session 8

- The Gradio demo can consume a generator like `stream_response(...)`.
- The Streamlit app can pass the same generator to `st.write_stream(...)`.
- Streaming is not a separate concept from app building; it is one of the main reasons the app feels interactive.